# MT-2 · Multi-Task Trainer — KL + OARSI Sub-Features

Trains the ordinal model to predict the KL grade together with the objective OARSI
sub-features (osteophyte, joint-space narrowing, sclerosis) as auxiliary ordinal
tasks. Sub-feature supervision exists only on OAI and only where each feature was
read, so every sub-grade loss is multiplied by its own per-knee mask. Learning these
physically-grounded features encourages a representation that transfers across
datasets better than one trained on the composite KL label alone.

The model, data loading, sampler, curriculum, domain-adversarial term, gradient
clipping, and crash-proof checkpointing follow the maximum-model library. Leave-one-
dataset-out evaluation is unchanged. Writes only to scope3_mt; nothing else is
touched. Includes the same anti-leakage and all-grades guards as the within-one work.

## Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, os
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import numpy as np, pandas as pd, json, glob, time, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
device='cuda' if torch.cuda.is_available() else 'cpu'
if device!='cuda': raise RuntimeError('No GPU.')

PROJECT=Path('/content/drive/MyDrive/Master Thesis')
MT_ROOT=PROJECT/'scope3_mt'; MT_CKPT=MT_ROOT/'checkpoints'; MT_RES=MT_ROOT/'results'
for d in (MT_ROOT,MT_CKPT,MT_RES): d.mkdir(parents=True, exist_ok=True)

MT_MANIFEST=MT_ROOT/'manifest_mt.csv'
assert MT_MANIFEST.exists(), 'Run MT-1 first to create manifest_mt.csv'
mt_man=pd.read_csv(str(MT_MANIFEST))
SUB=[c for c in ['osteophyte_max','jsn_max','sclerosis_max'] if c in mt_man.columns]
MASK=[c.replace('_max','_mask') for c in SUB]
print('Sub-features:', SUB)
print('Coverage (labelled knees):')
for c,m in zip(SUB,MASK): print(f'  {c}: {int(mt_man[m].sum()):,}')

Mounted at /content/drive
Sub-features: ['osteophyte_max', 'jsn_max', 'sclerosis_max']
Coverage (labelled knees):
  osteophyte_max: 8,547
  jsn_max: 8,547
  sclerosis_max: 8,547


## Load shared image array, attach sub-grades to TM's loader

The multi-task manifest carries the same `arr_idx` as the packed array, so images
load identically. The sub-grade values and masks are looked up per row during
batching.

In [3]:
manifest = TM.prepare_local_data()

cols_to_add = SUB+MASK
mt_idx = mt_man.set_index('arr_idx')[cols_to_add]
manifest = manifest.set_index('arr_idx')
for c in cols_to_add:
    manifest[c] = mt_idx[c]
manifest = manifest.reset_index()

for c in SUB: manifest[c]=manifest[c].fillna(-1.0)
for m in MASK: manifest[m]=manifest[m].fillna(0).astype(int)
manifest['patient_id'] = manifest['dataset'].astype(str)+'::'+manifest['filename'].astype(str).str.extract(r'(\d{5,})')[0].fillna('na')
print('Manifest ready with sub-grades. Rows:', len(manifest))
print('OAI rows with JSN label:', int(manifest[MASK[1]].sum()) if len(MASK)>1 else 'n/a')

Copied array in 35s
Loaded array (61558, 224, 224) in 1s
Manifest ready with sub-grades. Rows: 61558
OAI rows with JSN label: 8547


## Multi-task dataset and model

The dataset returns, alongside the image and KL label, a vector of sub-grade targets
and a matching mask vector. The model reuses the CORN backbone and KL head and adds a
small ordinal (CORN-style) head per sub-feature. Each sub-feature is graded 0..3, so
each head outputs three cumulative logits.

In [4]:
SUBK = 4
class MTDataset(torch.utils.data.Dataset):
    def __init__(self, df, train=True, quality=None):
        self.df=df.reset_index(drop=True); self.train=train; self.quality=quality or {}
    def __len__(self): return len(self.df)
    def _aug(self,a):
        if not self.train: return a
        import cv2, random
        if random.random()<0.5: a=np.fliplr(a).copy()
        if random.random()<0.5:
            ang=random.uniform(-7,7); h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),ang,1.0); a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
        if random.random()<0.3:
            f=random.uniform(0.9,1.1); a=np.clip(a.astype(np.float32)*f,0,255).astype(np.uint8)
        return a
    def __getitem__(self,i):
        r=self.df.iloc[i]; a=TM._resize(TM.joint_crop(TM._get_image(r))); a=self._aug(a)
        x=torch.from_numpy(a.astype(np.float32)/255.0); x=(x-0.485)/0.229; x=x.unsqueeze(0).repeat(3,1,1)
        y=int(r['kl_grade']); base=TM.LOSS_WEIGHTS.get(r['kl_source'],1.0)
        if r['dataset']=='mrkr': base=base*TM.MRKR_TRUST
        qw=self.quality.get(r['filename'],1.0) if r['kl_source']=='model_predicted' else 1.0
        sub=torch.tensor([max(0,int(r[c])) for c in SUB], dtype=torch.long)
        msk=torch.tensor([int(r[m]) for m in MASK], dtype=torch.float32)
        dom=TM.DATASET_TO_IDX.get(r['dataset'],0)
        return x, y, torch.tensor(base*qw,dtype=torch.float32), dom, sub, msk

class MultiTaskNet(nn.Module):
    def __init__(self, n_sub):
        super().__init__()
        self.core = TM.OrdinalNet(config.NUM_CLASSES, 4, use_hierarchical=True)
        feat = self.core.feat_dim
        self.sub_heads = nn.ModuleList([nn.Sequential(nn.Flatten(1),nn.LayerNorm(feat),nn.Dropout(0.3),nn.Linear(feat,SUBK-1)) for _ in range(n_sub)])
    def forward(self, x, grl_lambda=0.0):
        f=self.core.backbone(x)
        if f.dim()==4: f=f.mean(dim=[-2,-1])
        kl=self.core.corn(f); s1=self.core.head_s1(f); s2=self.core.head_s2(f)
        dom=self.core.domain_head(TM.grad_reverse(f,grl_lambda))
        subs=[h(f) for h in self.sub_heads]
        return kl, s1, s2, dom, subs
    def freeze_stages(self,n): self.core.freeze_stages(n)
    def param_groups(self):
        hp,bp=[],[]
        for n,p in self.named_parameters():
            if p.requires_grad: (hp if ('corn' in n or 'head' in n or 'sub_heads' in n) else bp).append(p)
        return hp,bp
print('Multi-task model defined: KL CORN head +', len(SUB), 'sub-feature heads.')

Multi-task model defined: KL CORN head + 3 sub-feature heads.


## Masked sub-feature loss

Each sub-feature uses the same conditional CORN loss as KL, but the per-sample loss
is multiplied by that feature's mask, so knees lacking a given reading contribute no
gradient for that head. The total objective is KL + a weighted sum of the present
sub-feature losses.

In [5]:
SUB_WEIGHT = 0.3

def masked_sub_loss(sub_logits_list, sub_targets, sub_mask):

    total=torch.tensor(0.0, device=sub_targets.device); active=0
    for j,logits in enumerate(sub_logits_list):
        y=sub_targets[:,j]; m=sub_mask[:,j]
        if m.sum()<1: continue
        tgt,tmask=TM.corn_targets(y, SUBK)
        bce=F.binary_cross_entropy_with_logits(logits,tgt,reduction='none')
        bce=(bce*tmask).sum(1)/tmask.sum(1).clamp(min=1)
        total=total + (bce*m).sum()/m.sum().clamp(min=1)
        active+=1
    return total/max(1,active)

def within1(yt,yp): return float((np.abs(np.asarray(yt)-np.asarray(yp))<=1).mean())
def pred_health(yp,n=5):
    c=np.bincount(np.asarray(yp),minlength=n); return int((c>0).sum())

@torch.no_grad()
def mt_predict(model, df, bs=12, use_tta=True):
    import cv2
    model.eval(); df=df.reset_index(drop=True); preds=[]; probs=[]
    def ln(a):
        x=torch.from_numpy(a.astype(np.float32)/255.0); x=(x-0.485)/0.229; return x.unsqueeze(0).repeat(3,1,1)
    tfs=TM.TTA_TRANSFORMS if use_tta else ['identity']
    for s in range(0,len(df),bs):
        sub=df.iloc[s:s+bs]; acc=None
        for tn in tfs:
            b=[]
            for _,r in sub.iterrows():
                a=TM._resize(TM.joint_crop(TM._get_image(r)))
                if tn=='hflip': a=np.fliplr(a).copy()
                elif tn=='rot+5':
                    h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),5,1.0); a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
                elif tn=='rot-5':
                    h,w=a.shape; M=cv2.getRotationMatrix2D((w/2,h/2),-5,1.0); a=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_REFLECT)
                b.append(ln(a))
            xb=torch.stack(b).to(device); kl,_,_,_,_=model(xb,grl_lambda=0.0)
            p=TM.corn_probs(kl).cpu().numpy(); acc=p if acc is None else acc+p
        acc/=len(tfs); probs.append(acc); preds.extend(acc.argmax(1).tolist())
    return np.array(preds), np.vstack(probs)
print('Masked sub-loss + multi-task predictor ready.')

Masked sub-loss + multi-task predictor ready.


## Guarded multi-task training (one fold)

In [6]:
def assert_no_leak(tr,te,ds):
    assert ds not in set(tr['dataset'].unique()), f'LEAK: {ds} in training!'
    ov=set(tr['patient_id'])&set(te['patient_id']); assert len(ov)==0, f'LEAK: {len(ov)} shared ids!'
    print(f'  Guard PASS: no leakage (train {len(tr)}, test {len(te)})')

def make_splits(man, test_ds, seed=0, val_frac=0.15):
    pool=man[man['dataset']!=test_ds].reset_index(drop=True)
    va=pool.sample(frac=val_frac, random_state=seed); tr=pool.drop(va.index)
    te=man[man['dataset']==test_ds].reset_index(drop=True)
    return tr.reset_index(drop=True), va.reset_index(drop=True), te

def train_mt(run_name, test_ds, seed=0, log_fn=print):
    TM.set_seed(seed); done=MT_RES/f'{run_name}.json'
    if done.exists(): log_fn(f'[{run_name}] exists'); return json.load(open(str(done)))
    tr,va,te=make_splits(manifest, test_ds, seed); assert_no_leak(tr,te,test_ds)
    quality=TM.load_quality_weights()
    model=MultiTaskNet(len(SUB)).to(device); model.freeze_stages(TM.MAX_FREEZE_STAGES)
    hp,bp=model.param_groups()
    opt=torch.optim.AdamW([{'params':hp,'lr':TM.MAX_LR_HEAD},{'params':bp,'lr':TM.MAX_LR_BACKBONE}],weight_decay=TM.WEIGHT_DECAY)
    scaler=torch.amp.GradScaler('cuda'); bs=TM.MAX_BATCH_SIZE; accum=TM.MAX_GRAD_ACCUM
    ckpt=MT_CKPT/f'{run_name}.pt'; ckb=MT_CKPT/f'{run_name}_best.pt'; e0,best,hist=0,0.0,[]
    if ckpt.exists():
        try: e0,best,hist=TM.load_ckpt(str(ckpt),model,opt); log_fn(f'[{run_name}] resume ep{e0}')
        except Exception: e0,best,hist=0,0.0,[]
    no_imp=0; from torch.utils.data import DataLoader
    for epoch in range(e0,TM.MAX_EPOCHS):
        if epoch<TM.CURRICULUM['phase1_end']: clean,mw=True,0.0
        elif epoch<TM.CURRICULUM['phase2_end']:
            clean=False; fr=(epoch-TM.CURRICULUM['phase1_end'])/max(1,TM.CURRICULUM['phase2_end']-TM.CURRICULUM['phase1_end']); mw=fr*TM.CURRICULUM['mrkr_target_weight']
        else: clean,mw=False,TM.CURRICULUM['mrkr_target_weight']
        loader=DataLoader(MTDataset(tr,True,quality),batch_size=bs,sampler=TM.build_sampler(tr,mw,clean),num_workers=TM.NUM_WORKERS,pin_memory=True)
        sc=(epoch+1)/TM.MAX_WARMUP if epoch<TM.MAX_WARMUP else 0.5*(1+math.cos(math.pi*(epoch-TM.MAX_WARMUP)/max(1,TM.MAX_EPOCHS-TM.MAX_WARMUP)))
        opt.param_groups[0]['lr']=TM.MAX_LR_HEAD*sc; opt.param_groups[1]['lr']=TM.MAX_LR_BACKBONE*sc
        grl=0.5*(2.0/(1.0+math.exp(-10*epoch/max(1,TM.MAX_EPOCHS)))-1.0)
        model.train(); t0=time.time(); rl=0.0; nb=0; tc=0; tt=0; opt.zero_grad()
        for bi,(x,y,lw,dom,sub,msk) in enumerate(loader):
            x,y,lw,dom=x.to(device),y.to(device),lw.to(device),dom.to(device)
            sub,msk=sub.to(device),msk.to(device)
            with torch.amp.autocast('cuda'):
                kl,s1,s2,dl,subs=model(x,grl_lambda=grl)
                loss=TM.corn_loss(kl,y,lw,TM.MAX_SOFT_ALPHA)+TM.hier_aux_loss(s1,s2,y)+TM.domain_loss(dl,dom)
                loss=loss + SUB_WEIGHT*masked_sub_loss(subs,sub,msk)
                loss=loss/accum
            scaler.scale(loss).backward()
            if (bi+1)%accum==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); opt.zero_grad()
            rl+=loss.item()*accum; nb+=1
            with torch.no_grad(): tc+=(TM.corn_predict(kl)==y).sum().item(); tt+=y.size(0)
        tra=tc/max(1,tt)
        vp,vpr=mt_predict(model,va,bs,use_tta=False); vm=TM.compute_metrics(va['kl_grade'].values,vp,vpr); vw1=within1(va['kl_grade'].values,vp); u=pred_health(vp)
        hist.append({'epoch':epoch,'loss':rl/max(1,nb),'train_acc':tra,'val_acc':vm['acc5'],'val_within1':vw1,'val_qwk':vm['qwk'],'val_grades':u})
        log_fn(f"  [{run_name}] ep{epoch+1}/{TM.MAX_EPOCHS} loss={rl/max(1,nb):.3f} tr={tra:.3f} val={vm['acc5']:.3f} w1={vw1:.3f} qwk={vm['qwk']:.3f} grades={u}/5 ({time.time()-t0:.0f}s)")
        score=vm['qwk'] if u>=4 else vm['qwk']*0.5
        if score>best:
            best=score
            try: TM.save_ckpt(str(ckb),model,opt,epoch,best,hist)
            except Exception: pass
            no_imp=0
        else: no_imp+=1
        try: TM.save_ckpt(str(ckpt),model,opt,epoch+1,best,hist)
        except Exception: pass
        if no_imp>=TM.MAX_PATIENCE: log_fn(f'  [{run_name}] early stop'); break
    if ckb.exists():
        try: TM.load_ckpt(str(ckb),model,None)
        except Exception: pass
    tp,tpr=mt_predict(model,te,bs,use_tta=TM.USE_TTA); tm=TM.compute_metrics(te['kl_grade'].values,tp,tpr); tw1=within1(te['kl_grade'].values,tp); used=pred_health(tp)
    res={'run_name':run_name,'test_ds':test_ds,'seed':seed,
         'external_acc5':tm['acc5'],'external_within1':tw1,'external_qwk':tm['qwk'],
         'external_binary':tm['acc_binary'],'external_auc_oa':tm.get('auc_oa',float('nan')),
         'grades_used':used,'internal_qwk':best}
    np.savez_compressed(str(MT_RES/f'{run_name}_preds.npz'),y_true=te['kl_grade'].values,y_pred=tp,probs=tpr)
    json.dump(res,open(str(done),'w'),indent=2)
    log_fn(f"  [{run_name}] DONE exact={tm['acc5']:.3f} w1={tw1:.3f} qwk={tm['qwk']:.3f} binary={tm['acc_binary']:.3f} aucOA={tm.get('auc_oa',float('nan')):.3f} grades={used}/5")
    return res
print('Guarded multi-task trainer ready.')

Guarded multi-task trainer ready.


## Run — single fold test first (OAI held out)

OAI is held out so the sub-feature supervision benefits the *other* datasets'
training and we test whether it improves transfer back to OAI. (Note: when OAI is the
test fold, sub-grades are only in the held-out set, so they aid via the shared
auxiliary signal during training on the other three — to directly use OAI sub-grades
in training, also run a fold where OAI is in the training pool, e.g. test=mendeley.)

In [7]:

res_fold = train_mt('mt_oai_seed0', 'oai', seed=0)

[mt_oai_seed0] exists


## Compare to baseline (MAX) on the same fold

In [8]:
def base_metrics(fold_glob):
    fs=sorted(glob.glob(str(config.RESULTS_DIR/fold_glob)))
    if not fs: return None
    z=np.load(fs[0]); from sklearn.metrics import cohen_kappa_score, accuracy_score
    yt,yp=z['y_true'],z['y_pred']
    return {'exact':float(accuracy_score(yt,yp)),'within1':within1(yt,yp),'qwk':float(cohen_kappa_score(yt,yp,weights='quadratic'))}

b=base_metrics('fold4_test_oai_seed*_preds.npz')
print('OAI fold — baseline H vs Multi-Task:')
print(f"{'metric':10s}{'H':>10s}{'Multi-Task':>12s}")
if b:
    print(f"{'exact':10s}{b['exact']:>10.3f}{res_fold['external_acc5']:>12.3f}")
    print(f"{'within1':10s}{b['within1']:>10.3f}{res_fold['external_within1']:>12.3f}")
    print(f"{'qwk':10s}{b['qwk']:>10.3f}{res_fold['external_qwk']:>12.3f}")
    print()
    print(f"QWK delta: {res_fold['external_qwk']-b['qwk']:+.3f}")
    print(f"Binary OA AUC: {res_fold['external_auc_oa']:.3f}")
print('\nIf QWK improved, the sub-feature signal is helping. Then run other folds.')

OAI fold — baseline H vs Multi-Task:
metric             H  Multi-Task
exact          0.484       0.518
within1        0.782       0.810
qwk            0.533       0.568

QWK delta: +0.035
Binary OA AUC: 0.875

If QWK improved, the sub-feature signal is helping. Then run other folds.
